# Music Annotation Project V5.4: Efficient Controlled Reconstruction Benchmark

**Runtime:** CPU.

This rewrite preserves the controlled benchmark while reducing repeated work:

- Telea, Navier–Stokes and biharmonic base reconstructions are each computed once per page.
- Both hybrid methods reuse those base reconstructions.
- Reconstruction and metric caches are persisted in Google Drive across Colab sessions.
- Page-level features and SSIM maps are computed once per method/page rather than once per region or component.
- Component metrics are evaluated on bounded crops rather than repeatedly processing the full page.
- Cache hits and misses, reconstruction time and metric time are reported per page.
- Pixels outside the mask remain immutable and are asserted.
- The clean page is used only for evaluation after reconstruction.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


## Configuration, helpers and persistent local staging


In [ ]:
import cv2, json, hashlib, shutil, math, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from skimage.metrics import structural_similarity as ssim
from skimage.restoration import inpaint as sk_inpaint

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
METHODS=['telea','navier_stokes','biharmonic','hybrid_conservative','hybrid_routed']
CACHE_SCHEMA=4
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
DRIVE_CLEAN=ROOT/'source'/'clean'
DRIVE_MASK=ROOT/'masks'/'corrected'
DRIVE_OUT=ROOT/'reconstruction_benchmark_v4'
DRIVE_CACHE=DRIVE_OUT/'persistent_cache'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=DRIVE_OUT/RUN_NAME
LOCAL=Path('/content/reconstruction_benchmark_v4')
LCLEAN=LOCAL/'clean'; LMASK=LOCAL/'masks'; LOCAL_CACHE=LOCAL/'cache'

CROP_MARGIN=48
OCCLUSION_VALUE=50
MIN_COMPONENT_AREA=20
MAX_FAILURE_CROPS=20
BOOTSTRAP_SAMPLES=2000
FORCE_REBUILD=False
FORCE_RECALCULATE_METRICS=False
KEEP_PAGES_IN_MEMORY=False

for d in [OUT,OUT/'tables',OUT/'failure_crops',OUT/'page_comparisons',
          DRIVE_CACHE,LCLEAN,LMASK,LOCAL_CACHE]:
    d.mkdir(parents=True,exist_ok=True)

def image_files(folder):
    folder=Path(folder)
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []

def maps(folder): return {p.stem:p for p in image_files(folder)}

def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def object_hash(value):
    return hashlib.md5(json.dumps(value,sort_keys=True).encode()).hexdigest()

def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0])
    return out

def atomic_copy(source,destination):
    destination=Path(destination); destination.parent.mkdir(parents=True,exist_ok=True)
    temp=destination.with_suffix(destination.suffix+'.tmp')
    shutil.copy2(source,temp); temp.replace(destination)

clean_map=maps(DRIVE_CLEAN); mask_map=maps(DRIVE_MASK)
pages=sorted(set(clean_map)&set(mask_map))
if not pages: raise RuntimeError('No matched clean/mask pages.')

# Copy only absent or changed source files to fast local storage.
for page in pages:
    for source,destination_folder in [(clean_map[page],LCLEAN),(mask_map[page],LMASK)]:
        destination=destination_folder/(page+source.suffix.lower())
        if not destination.exists() or source.stat().st_size!=destination.stat().st_size or source.stat().st_mtime_ns!=destination.stat().st_mtime_ns:
            shutil.copy2(source,destination)

print('Controlled pages:',len(pages))
print('Local staging:',LOCAL)
print('Persistent cache:',DRIVE_CACHE)
print('Durable output:',OUT)


## Crop grouping and base reconstruction


In [ ]:
def padded_components(mask,margin=40,min_area=8):
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    boxes=[]; h,w=mask.shape
    for i in range(1,n):
        x,y,bw,bh,area=stats[i]
        if area>=min_area:
            boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0); ay0=min(ay0,by0); ax1=max(ax1,bx1); ay1=max(ay1,by1); changed=True
                else:
                    rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def crop_inpaint_from_boxes(image,mask,method,boxes):
    result=image.copy()
    for x0,y0,x1,y1 in boxes:
        im=image[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
        if not np.any(mk): continue
        if method=='telea':
            candidate=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA)
        elif method=='navier_stokes':
            candidate=cv2.inpaint(im,mk,5,cv2.INPAINT_NS)
        elif method=='biharmonic':
            candidate=sk_inpaint.inpaint_biharmonic(im.astype(np.float32)/255.0,mk>0,channel_axis=None)
            candidate=np.clip(candidate*255,0,255).astype(np.uint8)
        else:
            raise ValueError(method)
        target=result[y0:y1,x0:x1]
        target[mk>0]=candidate[mk>0]
    return immutable(image,result,mask)

def staff_evidence(observed,mask):
    visible=observed.copy(); visible[mask>0]=255
    ink=(visible<190).astype(np.uint8)*255
    long=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))
    return cv2.dilate(long,np.ones((3,7),np.uint8))>0

def safe_staff_continuation(observed,mask,base,evidence=None,max_gap=80):
    out=base.copy(); evidence=staff_evidence(observed,mask) if evidence is None else evidence
    h,w=observed.shape
    for y in np.where(evidence.sum(axis=1)>=18)[0]:
        xs=np.where(mask[y]>0)[0]
        if not xs.size: continue
        for run in np.split(xs,np.where(np.diff(xs)>1)[0]+1):
            if not len(run) or len(run)>max_gap: continue
            x0,x1=int(run[0]),int(run[-1]); radius=16
            left=evidence[max(0,y-1):min(h,y+2),max(0,x0-radius):x0]
            right=evidence[max(0,y-1):min(h,y+2),x1+1:min(w,x1+radius+1)]
            if left.sum()<2 or right.sum()<2: continue
            values=[]
            for yy in range(max(0,y-1),min(h,y+2)):
                for xa,xb in [(max(0,x0-radius),x0),(x1+1,min(w,x1+radius+1))]:
                    q=observed[yy,xa:xb]; values.extend(q[q<190].tolist())
            if not values: continue
            value=min(int(np.percentile(values,35)),140)
            segment=mask[max(0,y-1):min(h,y+2),x0:x1+1]>0
            patch=out[max(0,y-1):min(h,y+2),x0:x1+1]
            patch[segment]=value
    return immutable(observed,out,mask)

def reconstruct_once(observed,mask):
    boxes=padded_components(mask,CROP_MARGIN)
    telea=crop_inpaint_from_boxes(observed,mask,'telea',boxes)
    navier=crop_inpaint_from_boxes(observed,mask,'navier_stokes',boxes)
    biharmonic=crop_inpaint_from_boxes(observed,mask,'biharmonic',boxes)
    evidence=staff_evidence(observed,mask)
    conservative=safe_staff_continuation(observed,mask,biharmonic,evidence)
    edge=cv2.dilate(cv2.Canny(observed,70,170),np.ones((3,3),np.uint8))>0
    routed=biharmonic.copy(); route=(mask>0)&edge&~evidence; routed[route]=navier[route]
    routed=safe_staff_continuation(observed,mask,immutable(observed,routed,mask),evidence)
    return {'telea':telea,'navier_stokes':navier,'biharmonic':biharmonic,
            'hybrid_conservative':conservative,'hybrid_routed':routed}


## Classification and efficient metric helpers


In [ ]:
def printed_ink(image):
    return cv2.adaptiveThreshold(image,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def edges(image): return cv2.Canny(image,70,170)>0

def classify_from_ink(ink,mask):
    long=cv2.morphologyEx(ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0
    staff_near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0
    symbol=ink&~staff_near; selected=mask>0
    return {'all':selected,'background':selected&~ink,'staff':selected&staff,
            'symbol':selected&symbol,'mixed':selected&staff_near&ink}

def f1_from_features(reference,prediction,region,tolerance=1):
    reference=reference&region; prediction=prediction&region
    if reference.sum()==0: return np.nan,np.nan,np.nan
    kernel=np.ones((2*tolerance+1,2*tolerance+1),np.uint8)
    ref_dilated=cv2.dilate(reference.astype(np.uint8),kernel)>0
    pred_dilated=cv2.dilate(prediction.astype(np.uint8),kernel)>0
    precision=float(np.sum(prediction&ref_dilated)/max(prediction.sum(),1))
    recall=float(np.sum(reference&pred_dilated)/reference.sum())
    return precision,recall,2*precision*recall/max(precision+recall,1e-12)

def feature_bundle(clean,reconstruction):
    _,ssim_map=ssim(clean,reconstruction,data_range=255,full=True)
    return {
        'difference':np.abs(clean.astype(np.float32)-reconstruction.astype(np.float32)),
        'ssim_map':ssim_map,
        'ink':printed_ink(reconstruction),
        'edges':edges(reconstruction),
    }

def metrics_from_bundle(clean,reconstruction,region,clean_ink,clean_edges,bundle):
    pixel_count=int(region.sum())
    if pixel_count==0: return None
    delta=clean[region].astype(np.float32)-reconstruction[region].astype(np.float32)
    mse=float(np.mean(delta*delta))
    ink_precision,ink_recall,ink_f1=f1_from_features(clean_ink,bundle['ink'],region,1)
    _,_,edge_f1=f1_from_features(clean_edges,bundle['edges'],region,1)
    return {'pixels':pixel_count,'mae':float(bundle['difference'][region].mean()),
            'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),
            'ssim':float(bundle['ssim_map'][region].mean()),
            'ink_precision':ink_precision,'ink_recall':ink_recall,'ink_f1':ink_f1,
            'edge_f1_tolerant':edge_f1}

def component_category(regions,component):
    counts={name:int(np.sum(region&component)) for name,region in regions.items() if name!='all'}
    nonzero={name:value for name,value in counts.items() if value>0}
    if not nonzero: return 'background'
    top=max(nonzero,key=nonzero.get)
    if len([value for value in nonzero.values() if value>=.2*component.sum()])>1: return 'mixed'
    return top


## Persistent cached controlled run


In [ ]:
rows=[]; components=[]; page_assets={}; timings=[]
for page_index,page in enumerate(pages,1):
    page_start=time.perf_counter()
    clean_path=next(LCLEAN.glob(page+'.*')); mask_path=next(LMASK.glob(page+'.*'))
    clean=cv2.imread(str(clean_path),cv2.IMREAD_GRAYSCALE)
    mask=cv2.imread(str(mask_path),cv2.IMREAD_GRAYSCALE)
    if clean is None or mask is None or clean.shape!=mask.shape:
        print('SKIPPED unreadable or mismatched:',page); continue
    mask=np.where(mask>0,255,0).astype(np.uint8)
    observed=clean.copy(); observed[mask>0]=OCCLUSION_VALUE
    clean_ink=printed_ink(clean); clean_edges=edges(clean); regions=classify_from_ink(clean_ink,mask)
    fingerprint=object_hash({'clean':md5(clean_path),'mask':md5(mask_path),'margin':CROP_MARGIN,
                             'occlusion':OCCLUSION_VALUE,'methods':METHODS,'schema':CACHE_SCHEMA})
    persistent_npz=DRIVE_CACHE/f'{page}_{fingerprint}_recon.npz'
    local_npz=LOCAL_CACHE/persistent_npz.name
    metric_cache=DRIVE_CACHE/f'{page}_{fingerprint}_metrics.json'

    reconstruction_start=time.perf_counter()
    if persistent_npz.exists() and not FORCE_REBUILD:
        if not local_npz.exists(): shutil.copy2(persistent_npz,local_npz)
        with np.load(local_npz) as archive:
            methods={method:archive[method] for method in METHODS}
        cache_status='HIT'
    else:
        methods=reconstruct_once(observed,mask)
        np.savez(local_npz,**methods) # uncompressed: faster read/write; Drive copy is durable
        atomic_copy(local_npz,persistent_npz)
        cache_status='MISS'
    reconstruction_seconds=time.perf_counter()-reconstruction_start

    # Verify the benchmark's strongest invariant before accepting cached or new results.
    for method,reconstruction in methods.items():
        assert reconstruction.shape==clean.shape
        assert np.array_equal(reconstruction[mask==0],observed[mask==0]),f'{page} {method} changed unmasked pixels'

    metric_start=time.perf_counter()
    if metric_cache.exists() and not FORCE_RECALCULATE_METRICS and not FORCE_REBUILD:
        cached=json.loads(metric_cache.read_text())
        rows.extend(cached['rows']); components.extend(cached['components'])
        metric_status='HIT'
    else:
        page_rows=[]; page_components=[]
        bundles={method:feature_bundle(clean,reconstruction) for method,reconstruction in methods.items()}
        for method,reconstruction in methods.items():
            for region_name,region in regions.items():
                values=metrics_from_bundle(clean,reconstruction,region,clean_ink,clean_edges,bundles[method])
                if values:
                    page_rows.append({'page':page,'method':method,'region':region_name,
                                      'outside_mask_drift':0.0,**values})
        n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
        for component_id in range(1,n):
            x,y,width,height,area=[int(value) for value in stats[component_id]]
            if area<MIN_COMPONENT_AREA: continue
            component=labels==component_id
            category=component_category(regions,component)
            # The bundle was calculated once for the page; only indexed reduction happens here.
            for method,reconstruction in methods.items():
                values=metrics_from_bundle(clean,reconstruction,component,clean_ink,clean_edges,bundles[method])
                if values:
                    page_components.append({'page':page,'component':component_id,'category':category,
                                            'method':method,'x':x,'y':y,'w':width,'h':height,**values})
        rows.extend(page_rows); components.extend(page_components)
        temp=metric_cache.with_suffix('.json.tmp')
        temp.write_text(json.dumps({'schema':CACHE_SCHEMA,'rows':page_rows,'components':page_components}))
        temp.replace(metric_cache)
        metric_status='MISS'
    metric_seconds=time.perf_counter()-metric_start

    if KEEP_PAGES_IN_MEMORY:
        page_assets[page]=(clean,observed,mask,methods,regions)
    total_seconds=time.perf_counter()-page_start
    timings.append({'page':page,'reconstruction_cache':cache_status,'metric_cache':metric_status,
                    'reconstruction_seconds':reconstruction_seconds,'metric_seconds':metric_seconds,
                    'total_seconds':total_seconds})
    print(f'[{page_index}/{len(pages)}] {page}: reconstruction {cache_status} {reconstruction_seconds:.2f}s; metrics {metric_status} {metric_seconds:.2f}s; total {total_seconds:.2f}s')

results=pd.DataFrame(rows); component_df=pd.DataFrame(components); timing_df=pd.DataFrame(timings)
results.to_csv(OUT/'tables'/'metrics_by_page_region.csv',index=False)
component_df.to_csv(OUT/'tables'/'component_metrics.csv',index=False)
timing_df.to_csv(OUT/'tables'/'page_timings.csv',index=False)
print('Benchmark rows:',len(results),'Component rows:',len(component_df))


## Summaries and paired bootstrap comparisons


In [ ]:
summary=results.groupby(['method','region'],as_index=False).agg(
    pages=('page','nunique'),pixels=('pixels','sum'),mae=('mae','mean'),psnr=('psnr','mean'),
    ssim=('ssim','mean'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),
    edge_f1_tolerant=('edge_f1_tolerant','mean'),outside_mask_drift=('outside_mask_drift','max'))
summary.to_csv(OUT/'tables'/'reconstruction_summary_by_region.csv',index=False)
summary[summary.region=='all'].to_csv(OUT/'reconstruction_summary.csv',index=False)

def bootstrap_delta(values,seed=42):
    values=np.asarray(values,float); values=values[np.isfinite(values)]
    if not len(values): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed)
    samples=np.empty(BOOTSTRAP_SAMPLES,np.float64)
    for index in range(BOOTSTRAP_SAMPLES):
        samples[index]=rng.choice(values,len(values),replace=True).mean()
    return float(values.mean()),float(np.percentile(samples,2.5)),float(np.percentile(samples,97.5))

paired=[]; baseline='biharmonic'
for region,metric,higher in [('background','mae',False),('staff','ink_recall',True),
                             ('symbol','edge_f1_tolerant',True),('mixed','edge_f1_tolerant',True),('all','ssim',True)]:
    subset=results[results.region==region].pivot(index='page',columns='method',values=metric)
    if baseline not in subset: continue
    for method in METHODS:
        if method==baseline or method not in subset: continue
        delta=(subset[method]-subset[baseline])*(1 if higher else -1)
        mean,low,high=bootstrap_delta(delta.dropna())
        paired.append({'region':region,'metric':metric,'method':method,'baseline':baseline,
                       'benefit_delta_mean':mean,'ci95_low':low,'ci95_high':high,
                       'wins':int((delta>0).sum()),'losses':int((delta<0).sum()),'ties':int((delta==0).sum())})
paired_df=pd.DataFrame(paired)
paired_df.to_csv(OUT/'tables'/'paired_method_deltas.csv',index=False)
display(summary); display(paired_df); display(timing_df)


## Category-specific failure crops


In [ ]:
def load_page_assets(page):
    if page in page_assets: return page_assets[page]
    clean_path=next(LCLEAN.glob(page+'.*')); mask_path=next(LMASK.glob(page+'.*'))
    clean=cv2.imread(str(clean_path),0); mask=np.where(cv2.imread(str(mask_path),0)>0,255,0).astype(np.uint8)
    observed=clean.copy(); observed[mask>0]=OCCLUSION_VALUE
    regions=classify_from_ink(printed_ink(clean),mask)
    fingerprint=object_hash({'clean':md5(clean_path),'mask':md5(mask_path),'margin':CROP_MARGIN,
                             'occlusion':OCCLUSION_VALUE,'methods':METHODS,'schema':CACHE_SCHEMA})
    archive_path=DRIVE_CACHE/f'{page}_{fingerprint}_recon.npz'
    with np.load(archive_path) as archive:
        methods={method:archive[method] for method in METHODS}
    return clean,observed,mask,methods,regions

rank_rules={'background':('mae',False),'staff':('ink_recall',True),
            'symbol':('edge_f1_tolerant',True),'mixed':('edge_f1_tolerant',True)}
for category,(metric,higher) in rank_rules.items():
    target=(component_df[(component_df.category==category)&(component_df.method=='hybrid_routed')]
            .dropna(subset=[metric]).sort_values(metric,ascending=higher).head(MAX_FAILURE_CROPS))
    folder=OUT/'failure_crops'/category; folder.mkdir(parents=True,exist_ok=True)
    for rank,row in target.reset_index(drop=True).iterrows():
        clean,observed,mask,methods,regions=load_page_assets(row.page)
        height,width=clean.shape
        x0=max(0,int(row.x)-28); y0=max(0,int(row.y)-28)
        x1=min(width,int(row.x+row.w)+28); y1=min(height,int(row.y+row.h)+28)
        panels=[clean[y0:y1,x0:x1],observed[y0:y1,x0:x1]]+[methods[m][y0:y1,x0:x1] for m in METHODS]
        names=['ground truth','occlusion']+METHODS
        fig,axes=plt.subplots(1,len(panels),figsize=(4*len(panels),4))
        for axis,image,name in zip(axes,panels,names):
            axis.imshow(image,cmap='gray',vmin=0,vmax=255); axis.set_title(name); axis.axis('off')
        fig.suptitle(f'{category} | {row.page} component {int(row.component)} | {metric}={row[metric]:.3f}')
        plt.tight_layout(); plt.savefig(folder/f'{rank+1:03d}_{row.page}_{int(row.component):03d}.png',dpi=170,bbox_inches='tight'); plt.close(fig)
print('Failure crops:',OUT/'failure_crops')


## Manifest and cache report


In [ ]:
manifest={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'pages':len(pages),'methods':METHODS,'cache_schema':CACHE_SCHEMA,
    'guarantees':{'clean_used_only_for_scoring':True,'outside_mask_immutable':True,
                  'crop_only_reconstruction':True,'local_staging':True,
                  'persistent_reconstruction_cache':True,'persistent_metric_cache':True},
    'optimisations':{'base_methods_computed_once':True,'hybrids_reuse_base_methods':True,
                     'ssim_map_once_per_page_method':True,'page_features_reused':True,
                     'component_metrics_use_precomputed_features':True},
    'timing_table':str(OUT/'tables'/'page_timings.csv')
}
(OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Output:',OUT)
print('Persistent cache:',DRIVE_CACHE)
print('Key files:',OUT/'reconstruction_summary.csv',OUT/'tables'/'reconstruction_summary_by_region.csv',OUT/'tables'/'paired_method_deltas.csv',OUT/'tables'/'page_timings.csv')
